In [ ]:
# Necessary imports
import os
import pandas as pd
import numpy as np

# CANCER CONFIGURATION: edit for a new cancer
CANCER     = "KIRC"

# folder holding this cancer's run
base_dir   = r"causal_inf_new"

# this cancer's own folder: holds <prefix>_data, <prefix>_notebooks and <prefix>_output
cancer_dir = os.path.join(base_dir, CANCER.lower())

# this cancer's input data (expression files, downloaded clinical / validation cohorts)
data_dir   = os.path.join(cancer_dir, f"{CANCER.lower()}_data")
prefix     = CANCER.lower()
output_dir = os.path.join(cancer_dir, f"{prefix}_output")

# survival endpoint: "OS" (overall survival) or "DFS" (disease-free; use for THCA / PRAD, which have very few deaths)
ENDPOINT   = "OS"

# cBioPortal TCGA study for each cancer ; colon cancer is combined with rectal in cBioPortal
CANCER_SETTINGS = {
    "BRCA": "brca_tcga", "KIRC": "kirc_tcga", "LUAD": "luad_tcga", "UCEC": "ucec_tcga", "THCA": "thca_tcga",
    "PRAD": "prad_tcga", "LUSC": "lusc_tcga", "HNSC": "hnsc_tcga", "COAD": "coadread_tcga", "LGG": "lgg_tcga",
}

CBIO_STUDY = CANCER_SETTINGS[CANCER]

# clinical file from the cBioPortal study download, same nested layout as brca_tcga/brca_tcga/
clin_path = os.path.join(data_dir, CBIO_STUDY, CBIO_STUDY, "data_clinical_patient.txt")

os.makedirs(output_dir, exist_ok=True)
assert os.path.exists(clin_path), f"missing clinical file: {clin_path}"
print(f"Cancer: {CANCER} | study: {CBIO_STUDY} | endpoint: {ENDPOINT}")

In [ ]:
# cBioPortal files start with a few lines beginning with '#' (display names, descriptions, types, priority);
# skipping them leaves the column IDs as the header: PATIENT_ID, OS_STATUS, OS_MONTHS, DFS_STATUS, DFS_MONTHS, ...

n_skip = 0

with open(clin_path, encoding="utf-8") as f:
    for line in f:
        if line.startswith("#"):
            n_skip += 1
        else:
            break

# Skipping metadata rows and loading
clinical_raw = pd.read_csv(clin_path, sep="\t", skiprows=n_skip)

# Selecting the appropriate columns and renaming them
clinical_filtered = clinical_raw[["PATIENT_ID", "OS_STATUS", "OS_MONTHS"]].copy()

# Converting survival status to binary (1/0)
clinical_filtered["OS_STATUS"] = clinical_filtered["OS_STATUS"].map({"0:LIVING": 0, "1:DECEASED": 1})
clinical_filtered["OS_MONTHS"] = pd.to_numeric(clinical_filtered["OS_MONTHS"], errors="coerce")

# Dropping NaN and negative survival times
clinical_clean = clinical_filtered.dropna(axis = 0, how = 'any')
clinical_clean = clinical_clean[clinical_clean["OS_MONTHS"] >= 0]

# Set index:
clinical_clean = clinical_clean.set_index("PATIENT_ID")

print(clinical_filtered.shape, clinical_clean.shape)

clinical_clean.to_csv(os.path.join(output_dir, "clinical_data.csv"))
print(f"Saved clinical_data.csv: {clinical_clean.shape} | events: {clinical_clean['OS_STATUS'].sum()} "
      f"({100 * clinical_clean['OS_STATUS'].mean():.1f}%) | endpoint: {ENDPOINT}")